# 3. Entity Resolution (AI Auto-Merge)

Notebook ini menjalankan algoritma **Entity Resolution** menggunakan Vector Similarity dan fungsi APOC (`apoc.refactor.mergeNodes`). 

Tujuannya adalah menemukan *node* yang secara semantik sangat mirip (misal: "Oli Bocor" dan "Kebocoran Oli") dan menggabungkannya menjadi satu entitas. Hal ini akan mengurangi duplikasi dan memperkuat jaring-jaring graf (Graph Density) sebelum algoritma *Community Detection* dijalankan.

In [ ]:
import os
import sys

# Tambahkan root direktori ke path agar bisa import dari src
sys.path.append(os.path.abspath('..'))

from src.graph.client import GraphClient
from src.config import settings
from src.graph.entity_resolution import EntityResolver
from src.graph.index_manager import IndexManager

import logging
logging.basicConfig(level=logging.INFO)

In [ ]:
client = GraphClient(
    uri=settings.neo4j_uri,
    user=settings.neo4j_user,
    password=settings.neo4j_password
)

print("Memastikan Neo4j Vector Indexes sudah siap...")
manager = IndexManager(client)
manager.setup_all_indexes()
print("\nIndeks pencarian vektor siap!")

## Jalankan Proses Resolusi Entitas

Kita menggunakan `similarity_threshold = 0.95` yang berarti hanya *node* yang keakuratan maknanya di atas 95% yang akan digabung. Anda bisa menurunkannya menjadi `0.90` jika ingin penggabungan yang lebih agresif.

In [ ]:
resolver = EntityResolver(client)

# Jalankan untuk seluruh label pola (Symptom, Cause, Action, ProblemCluster)
total_merged = resolver.resolve_all(similarity_threshold=0.95)

print(f"\n\u2705 Selesai! Sebanyak {total_merged} node duplikat telah berhasil digabungkan.")

In [ ]:
client.close()